# STAGE E/5 — **v3** inference + **SWEEP abstention**

**Add Input (BẮT BUỘC):** **Stage C** (ner + assert) + **Stage D** (kb).
**Settings:** GPU **T4 (1 con)**. **~8′**. 1 lần chạy → NHIỀU zip ở các ngưỡng.

**output_ner_p00** = min_prob 0 (= bản 34.40). **p60/p75/p85** = cắt span thừa dần (đòn master: over-prediction phạt 3 tầng). **output_ctx** = ConText để A/B assertion. Nộp từng zip, tìm điểm cao nhất.


In [ ]:
# Clone repo
import os, subprocess
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # 1 GPU: tránh DataParallel chậm
REPO = "https://github.com/Khanhhh239/Medical_Retrieve.git"
D = "/kaggle/working/Medical_Retrieve"
if not os.path.isdir(D):
    subprocess.run(["git","clone","-q",REPO,D], check=True)
else:
    subprocess.run(["git","-C",D,"pull","-q"])
os.chdir(D); print('cwd:', os.getcwd())


In [ ]:
!pip install -q rapidfuzz pyyaml regex accelerate sentencepiece


In [ ]:
# Tìm artifact từ stage trước (đã Add Input)
import glob, os, shutil
def find_input(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    return hits[0] if hits else None
def find_model(name):
    # model dir = thư mục có config.json (TRÁNH khớp nhầm src/ner là code)
    hits = [h for h in glob.glob(f"/kaggle/input/**/{name}/config.json", recursive=True)
            if "/src/" not in h]
    return os.path.dirname(hits[0]) if hits else None


In [ ]:
import glob, os, shutil
NER = find_model('ner')
ASSERT = find_model('assert')
KB = find_input('kb')
if KB:                                   # nạp KB cho linker
    for f in glob.glob(KB + '/*'):
        if os.path.isfile(f): shutil.copy(f, 'data/kb/' + os.path.basename(f))
print('NER:', NER, '| ASSERT:', ASSERT, '| KB:', bool(KB))
assert NER, 'CHUA ADD INPUT Stage C (ner)!'


In [ ]:
# SWEEP abstention: 1 lần chạy model -> nhiều zip ở các ngưỡng (đòn cắt span thừa)
_am = f'--assert_model {ASSERT}' if ASSERT else ''
!python scripts/sweep_prob.py --model_dir {NER} {_am} --thresholds 0,0.6,0.75,0.85


In [ ]:
# A/B assertion: ConText (rule) ở min_prob=0 -> so với Head B
!python scripts/predict.py --pipeline ner --model_dir {NER} --out_dir output_ctx --zip


In [ ]:
# Copy tất cả zip ra /kaggle/working
import shutil, glob, os
for z in sorted(glob.glob('output_ner_p*.zip')) + ['output_ctx.zip']:
    if os.path.exists(z): shutil.copy(z, '/kaggle/working/'+os.path.basename(z)); print('SAVED', z)
print(">>> Nộp lần lượt p00/p60/p75/p85 tìm τ tốt nhất; output_ctx để so assertion <<<")


**Đọc log sweep:** in ra `concept/file` mỗi ngưỡng. Gold lâm sàng ~15-20/file → chọn τ kéo về khoảng đó. Nộp từng zip, τ cho điểm cao nhất là đáp án.
